# Local Qwen test (LM Studio)
Requires LM Studio server running on `localhost:1234`.

In [1]:
from openai import OpenAI
import time
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio", timeout=300)
MODEL = "qwen3-coder-30b-a3b-instruct"
[m.id for m in client.models.list().data]

['qwen3-vl-30b-a3b-instruct', 'qwen3-coder-30b-a3b-instruct']

## Chat

In [2]:
r = client.chat.completions.create(model=MODEL, max_tokens=200, temperature=0.2,
    messages=[{"role":"user","content":"Explain RAG in two sentences."}])
print(r.choices[0].message.content); print(r.usage)

Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with language generation by first searching through a document collection to find relevant passages, then using those retrieved results as context to generate more accurate and informed responses. This approach helps large language models provide more factual and up-to-date answers by grounding their responses in specific external knowledge rather than relying solely on their pre-trained parameters.
CompletionUsage(completion_tokens=80, prompt_tokens=16, total_tokens=96, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None)


## Streaming + speed

In [3]:
t=time.perf_counter(); n=0
for ch in client.chat.completions.create(model=MODEL, max_tokens=200, stream=True,
        messages=[{"role":"user","content":"Write a haiku about GPUs."}]):
    if ch.choices and ch.choices[0].delta.content:
        print(ch.choices[0].delta.content, end="", flush=True); n+=1
print(); print(f"{n} chunks in {time.perf_counter()-t:.1f}s")

Here's a haiku about GPUs:

Parallel threads dance
Streaming through silicon dreams—
Power meets precision
21 chunks in 1.8s


## Code generation

In [4]:
r = client.chat.completions.create(model=MODEL, max_tokens=400, temperature=0,
    messages=[{"role":"user","content":"Write a Python is_prime(n) function with a docstring. Code only."}])
print(r.choices[0].message.content)

```python
def is_prime(n):
    """
    Determine if a number is prime.
    
    A prime number is a natural number greater than 1 that has no positive 
    divisors other than 1 and itself.
    
    Args:
        n (int): The number to check for primality
        
    Returns:
        bool: True if the number is prime, False otherwise
        
    Examples:
        >>> is_prime(2)
        True
        >>> is_prime(17)
        True
        >>> is_prime(4)
        False
        >>> is_prime(1)
        False
    """
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
    
    return True
```


## Tool calling

In [5]:
tools=[{"type":"function","function":{"name":"get_weather","description":"Get weather for a city",
  "parameters":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}}}]
r = client.chat.completions.create(model=MODEL, tools=tools,
    messages=[{"role":"user","content":"What's the weather in Paris?"}])
print(r.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='iezmgryMiuYSv7XOS9bNBL8OjP9FCq0P', function=Function(arguments='{"city":"Paris"}', name='get_weather'), type='function')]
